# 🤖 Notebook 02: Model Training, Evaluation & Explainability
## AI Health Risk Predictor — Cardiovascular Disease

This notebook covers:
1. **Model Training** — Logistic Regression, XGBoost, Neural Net
2. **Discrimination Metrics** — AUC-ROC, AUC-PR
3. **Calibration** — Brier Score, Reliability Diagrams
4. **SHAP Explainability** — Global and per-patient explanations
5. **Fairness Analysis** — Subgroup performance by sex & ethnicity
6. **Model Comparison** — Head-to-head summary table


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100
print('Setup complete ✓')

## 1. Load Preprocessed Data

In [ ]:
from src.preprocessor import run_preprocessing_pipeline

splits = run_preprocessing_pipeline(
    input_csv='../data/synthetic_ehr.csv',
    output_dir='../data/splits',
    preprocessor_path='../models/preprocessor.pkl'
)

X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
test_raw = splits['test_raw']

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

## 2. Train Models

In [ ]:
from src.models import get_model

models = {}
probas = {}

for name in ['logistic_regression', 'xgboost', 'neural_net']:
    print(f'Training {name}...')
    m = get_model(name)
    if name == 'xgboost':
        m.fit(X_train, y_train, X_val=X_val, y_val=y_val)
    else:
        m.fit(X_train, y_train)
    models[name] = m
    probas[name] = m.predict_proba(X_test)
    print(f'  ✓ {name} trained')

print('\nAll models trained!')

## 3. Discrimination Metrics

In [ ]:
from src.evaluator import compute_metrics, print_metrics_table

all_metrics = {}
for name, proba in probas.items():
    m = compute_metrics(y_test.values, proba, n_bootstrap=200)
    all_metrics[name] = m
    lo, hi = m['auc_roc_ci']
    print(f'{name:25s}  AUC-ROC: {m["auc_roc"]:.3f} [{lo:.3f}–{hi:.3f}]  '
          f'AUC-PR: {m["auc_pr"]:.3f}  Brier: {m["brier_score"]:.4f}')

metrics_df = print_metrics_table(all_metrics)
display(metrics_df)

In [ ]:
# ROC curves
from src.evaluator import plot_roc_curves, plot_pr_curves, plot_calibration_curves

y_trues = {name: y_test.values for name in models}
plot_roc_curves(all_metrics, y_trues, probas, save_path='../reports/figures/roc_curves.png')
img = plt.imread('../reports/figures/roc_curves.png')
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Precision-Recall curves
plot_pr_curves(y_trues, probas, all_metrics, save_path='../reports/figures/pr_curves.png')
img = plt.imread('../reports/figures/pr_curves.png')
plt.imshow(img); plt.axis('off'); plt.show()

In [ ]:
# Calibration curves
plot_calibration_curves(y_trues, probas, save_path='../reports/figures/calibration.png')
img = plt.imread('../reports/figures/calibration.png')
plt.imshow(img); plt.axis('off'); plt.show()

## 4. SHAP Explainability

In [ ]:
from src.explainer import RiskExplainer

# Use XGBoost (TreeSHAP — fast and exact)
bg_sample = X_train.sample(200, random_state=42)
explainer = RiskExplainer(models['xgboost'], 'xgboost', bg_sample)

# Global importance bar
importance = explainer.plot_feature_importance_bar(
    X_test.sample(300, random_state=42),
    save_path='../reports/figures/shap_importance_bar.png'
)
img = plt.imread('../reports/figures/shap_importance_bar.png')
plt.figure(figsize=(9, 4)); plt.imshow(img); plt.axis('off'); plt.show()

print('Top 10 features:')
display(importance.head(10).to_frame('Mean |SHAP|').round(5))

In [ ]:
# Per-patient waterfall chart (sample patient)
sample = X_test.iloc[[5]]
fig = explainer.plot_waterfall(sample, save_path='../reports/figures/shap_waterfall_nb.png')
img = plt.imread('../reports/figures/shap_waterfall_nb.png')
plt.figure(figsize=(9, 5)); plt.imshow(img); plt.axis('off'); plt.show()

# Top features for this patient
top = explainer.get_top_features(sample, n=5)
for i, f in enumerate(top, 1):
    print(f"{i}. {f['feature']:30s} SHAP={f['shap_value']:+.4f}  {f['direction']}")

## 5. Fairness Analysis

In [ ]:
from src.evaluator import evaluate_fairness, plot_fairness_heatmap

fairness_df = evaluate_fairness(test_raw, probas['xgboost'])
display(fairness_df)

if len(fairness_df) > 0:
    plot_fairness_heatmap(fairness_df, save_path='../reports/figures/fairness_heatmap.png')
    img = plt.imread('../reports/figures/fairness_heatmap.png')
    plt.figure(figsize=(10, 3)); plt.imshow(img); plt.axis('off'); plt.show()

## 6. Save Models

In [ ]:
Path('../models').mkdir(exist_ok=True)

for name, model in models.items():
    model.save(f'../models/{name}.pkl')
    print(f'Saved: ../models/{name}.pkl')

joblib.dump(explainer, '../models/explainer_xgboost.pkl')
print('Saved: ../models/explainer_xgboost.pkl')

print('\n✓ All models saved!')

## 7. Risk Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = {'logistic_regression': '#3498db', 'xgboost': '#e74c3c', 'neural_net': '#2ecc71'}

for ax, (name, proba) in zip(axes, probas.items()):
    ax.hist(proba[y_test==0], bins=40, alpha=0.6, color='steelblue', label='No Event', density=True)
    ax.hist(proba[y_test==1], bins=40, alpha=0.6, color='firebrick', label='CVD Event', density=True)
    ax.axvline(proba.mean(), color='black', ls='--', lw=1.2, label=f'Mean={proba.mean():.2f}')
    ax.set_title(name.replace('_', ' ').title())
    ax.set_xlabel('Predicted Risk Score')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig.suptitle('Predicted Risk Score Distributions by True Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('✓ Notebook complete!')
print('\nNext steps:')
print('  1. Run the Streamlit app: streamlit run app/streamlit_app.py')
print('  2. Review fairness_results.csv for subgroup disparities')
print('  3. Check reports/final_report.md for full analysis')